In [48]:
import yt_dlp

In [49]:
import isodate

In [158]:
import json

In [50]:
from pydub import AudioSegment

In [51]:
from sarvamai import SarvamAI

In [52]:
import os

In [53]:
from dotenv import load_dotenv

load_dotenv()

True

In [54]:
client_sarvam = SarvamAI(api_subscription_key = os.getenv("SARVAM_API_KEY"))

In [229]:
client_youtube = os.getenv("YOUTUBE_API_KEY")

In [56]:
import tempfile

In [57]:
import anthropic

In [58]:
client_anthropic = anthropic.Anthropic(api_key = os.getenv('CLAUDE_API_KEY'))

In [59]:
def chunk_audio(audio_path, chunk_duration_ms=29000):
    """Split audio into 29s chunks to stay under Sarvam's limit."""
    audio = AudioSegment.from_file(audio_path)
    chunks = []

    for i, start in enumerate(range(0, len(audio), chunk_duration_ms)):
        chunk = audio[start:start + chunk_duration_ms]
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        chunk.export(tmp.name, format="mp3")
        chunks.append(tmp.name)
        tmp.close()

    return chunks

In [60]:
def transcribe_audio(audio_path):
    response = client_sarvam.speech_to_text.transcribe(
        file=open(audio_path, "rb"),
        model="saaras:v3",
        mode="transcribe"
    )
    return response

In [61]:
def transcribe_long_audio(audio_path):
    chunks = chunk_audio(audio_path)
    full_transcript = []

    try:
        for i, chunk_path in enumerate(chunks):
            print(f"Transcribing chunk {i+1}/{len(chunks)}...")
            result = transcribe_audio(chunk_path)
            full_transcript.append(result.transcript)  # adjust key based on Sarvam response structure
    finally:
        # Clean up temp chunk files
        for chunk_path in chunks:
            os.remove(chunk_path)

    return " ".join(full_transcript)

In [22]:
transcript = transcribe_long_audio("/Users/Rakshit.Lodha/Desktop/audio.mp3")

transcript

Transcribing chunk 1/2...
Transcribing chunk 2/2...


"You started your company to build something real, not to drown in admin work. There's a better way to run it now. People spend months watching YouTube videos, trying random AI tools, figuring it out alone, and most of them end up right where they started. I compressed all of that into two days chat GPT and AI workshop, learned 25 plus AI tools. You build real automations, real workflows, real systems, no coding required, over 10. 10 million people have done this. Next batch this weekend, link below."

In [62]:
def download_audio(url):
    """Works for both Instagram and YouTube URLs via yt-dlp."""
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tmp_path = tmp.name
    tmp.close()

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': tmp_path.replace(".mp3", ""),   # yt-dlp appends extension
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    # yt-dlp saves as <path>.mp3, but tmp_path already ends in .mp3
    actual_path = tmp_path.replace(".mp3", "") + ".mp3"
    return actual_path

In [63]:
def process_urls(url_list):
    """
    Takes a list of Instagram/YouTube URLs.
    Returns a dict: { url -> transcript }
    """
    results = {}

    for i, url in enumerate(url_list):
        print(f"\n[{i+1}/{len(url_list)}] Processing: {url}")
        audio_path = None

        try:
            print("  Downloading audio...")
            audio_path = download_audio(url)

            print("  Transcribing...")
            transcript = transcribe_long_audio(audio_path)
            results[url] = transcript

        except Exception as e:
            print(f"Failed: {e}")
            results[url] = None

        finally:
            if audio_path and os.path.exists(audio_path):
                os.remove(audio_path)      # clean up downloaded file

    return results

In [28]:
process_urls(instagram_urls)


[1/4] Processing: https://www.instagram.com/p/DVQoG75DN8Z/?igsh=MTdzd3FhbWFxYWFjdQ==
  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[2/4] Processing: https://www.instagram.com/p/DVQoG75DN8Z/?igsh=MTdzd3FhbWFxYWFjdQ==
  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[3/4] Processing: https://www.instagram.com/reel/DRH4veBjY0D/?igsh=aG1yczVvaHRtbzZt
  Transcribing...                                          
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[4/4] Processing: https://www.instagram.com/reel/DRJwUlOjVNR/?igsh=MXVyemM5dWFlbmpmdA==
  Transcribing...                                          
Transcribing chunk 1/3...
Transcribing chunk 2/3...
Transcribing chunk 3/3...


{'https://www.instagram.com/p/DVQoG75DN8Z/?igsh=MTdzd3FhbWFxYWFjdQ==': "You started your company to build something real, not to drown in admin work. There's a better way to run it now. People spend months watching YouTube videos, trying random AI tools, figuring it out alone, and most of them end up right where they started. I compressed all of that into two days chat GPT and AI workshop, learned 25 plus AI tools. You build real automations, real workflows, real systems, no coding required, over 10. 10 million people have done this. Next batch this weekend, link below.",
 'https://www.instagram.com/reel/DRH4veBjY0D/?igsh=aG1yczVvaHRtbzZt': "What if I told you rich Indian families secretly marry only inside four zodiac signs? India's wedding economy moves 10.7 lakh crore rupees every single year, one of the biggest private wealth transfer systems in the country. Every major astrology portal in India pushes the same categories: top signs for wealth, zodiac signs, born rich, money magnet

In [64]:
def analyze_references(urls):
    """Transcribe all URLs and extract style/structure patterns."""
    
    print("Transcribing reference videos...")
    transcripts = process_urls(urls)
    valid = {k: v for k, v in transcripts.items() if v}
    
    if not valid:
        print("No valid transcripts found.")
        return None

    reference_text = ""
    for url, transcript in valid.items():
        reference_text += f"---\nReference ({url}):\n{transcript}\n"

    print(f"\nAnalyzing {len(valid)} transcripts...")
    print(transcripts)

    prompt = f"""
You are analyzing short-form video scripts to extract creator style patterns.

Here are {len(valid)} transcripts:

{reference_text}

Extract and return a structured analysis with these sections:

1. HOOK PATTERNS
   - How do they open? (question / bold claim / stat / story?)
   - First sentence examples from the references

2. STRUCTURE
   - How is the middle organized? (list / story arc / contrast / problem-solution?)
   - Average number of points made

3. TONE & LANGUAGE
   - Formal or conversational?
   - Sentence length (short punchy vs longer flowing)
   - Any recurring phrases or patterns

4. CLOSING & CTA
   - How do they end? (soft CTA / hard sell / cliffhanger?)
   - Examples from references

5. PACING
   - Estimated words per segment
   - Total script length pattern

Be specific and pull examples from the actual transcripts.
"""

    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )

    analysis = response.content[0].text

    # Save analysis for reuse
    with open("reference_analysis.txt", "w") as f:
        f.write(analysis)
    print("\nAnalysis saved to reference_analysis.txt")

    return analysis

In [64]:
urls = ["https://youtube.com/shorts/9LOINWY_8TY?si=nPfU9hz4a4akcPta"]

In [65]:
analysis = analyze_references(urls)

Transcribing reference videos...

[1/1] Processing: https://youtube.com/shorts/9LOINWY_8TY?si=nPfU9hz4a4akcPta


  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

Analyzing 1 transcripts...
{'https://youtube.com/shorts/9LOINWY_8TY?si=nPfU9hz4a4akcPta': "Let me tell you the story of a fruit seller's son who started 40 years ago with absolutely nothing in his hand and today has built a ₹453 crore revenue generating business with 160 retail outlets. I'm talking about Raghunandan Kamat, the founder of Naturals Ice Creams. Naturals Ice Cream is rated as one of the most trusted brands in the country today, and he started from scratch and for 40 years has maintained consistency in quality and consistency in the experience for his customers. And when asked what is the secret sauce, was behind that consistency at such a big scale. He said that as a child when he used to go back home in the evenings, he had no source of entertainment. He had no television or mobile phone in his era. So the only thing he would do is he would observe the cooking me

In [65]:
def generate_script(topic, insight, analysis=None):
    """
    Generate a script using topic + insight.
    Optionally pass in saved analysis from Step 1.
    """

    # Load from file if not passed directly
    if analysis is None:
        try:
            with open("reference_analysis.txt", "r") as f:
                analysis = f.read()
            print("Loaded analysis from reference_analysis.txt")
        except FileNotFoundError:
            print("No analysis found — generating without style reference")
            analysis = "No reference analysis available. Use generic short-form video best practices."

    prompt = f"""
You are a short-form video script writer.

Here is a style analysis of reference videos from top creators:

{analysis}

Now write a NEW script using this style as a template.

TOPIC: {topic}
CORE INSIGHT: {insight}

Script structure:
1. HOOK (3-5 seconds) — stop the scroll immediately
2. CONTEXT (5 seconds) — why this matters to the viewer
3. CORE CONTENT (3-5 punchy points that deliver the insight)
4. CTA (5 seconds) — what should the viewer do next

Rules:
- Match the tone and pacing from the style analysis
- Total length: 200-250 words (fits ~90 seconds)
- No filler phrases like "In today's video..." or "Don't forget to like"
- Output the script ONLY — no labels, no commentary
"""

    response = client_anthropic.messages.create(
        model="claude-opus-4-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )

    script = response.content[0].text

    # Save script
    safe_topic = topic[:30].replace(" ", "_")
    filename = f"script_{safe_topic}.txt"
    with open(filename, "w") as f:
        f.write(script)
    print(f"Script saved to {filename}")

    return script

In [45]:
topic = "Why most founders waste money on AI tools"
insight = "Most founders buy AI tools to look innovative, not to solve a real workflow problem"

script = generate_script(topic, insight)
print("\n" + "="*50)
print("GENERATED SCRIPT")
print("="*50)
print(script)

Loaded analysis from reference_analysis.txt
Script saved to script_Why_most_founders_waste_money_.txt

GENERATED SCRIPT
You're not buying AI tools. You're buying the *feeling* of being innovative.

Here's what I see every week: founders dropping $500, $800, sometimes $2,000 a month on AI subscriptions they barely open. ChatGPT Plus, Jasper, Copy.ai, Notion AI, six different image generators, three video tools. The full stack. Looks impressive in the budget spreadsheet.

But here's the problem—none of it connects to an actual workflow.

No automation. No system. No process that runs without you babysitting it.

You bought tools. You didn't solve problems.

The founders actually winning with AI? They start backwards. They identify one bottleneck—customer support taking 4 hours daily, content repurposing eating weekends, lead qualification drowning the sales team. Then they find the tool that fixes *that specific thing*.

One tool. One workflow. One result you can measure.

The rest? It's

In [66]:
from googleapiclient.discovery import build

In [67]:
from datetime import datetime, timedelta

In [68]:
API_KEY = client_youtube

In [69]:
youtube = build("youtube", "v3", developerKey=API_KEY)

In [70]:
def find_channel_id(handle):
    response = youtube.channels().list(
        forHandle = handle,
        part = "id, snippet"
    ).execute()
    
    return response['items'][0]['id']

In [71]:
find_channel_id("@Rajiv.Talreja")

'UCYEtmpGBd3jEO1A8sJ4OEMg'

In [130]:
def get_videos_data(channel_id, days = 60):
    # youtube = build("youtube", "v3", developerKey=API_KEY)
    after = (datetime.utcnow() - timedelta(days=days)).isoformat("T") + "Z"

    full_data = []

    search = youtube.search().list(
        channelId = channel_id,
        publishedAfter = after, 
        type="video", order="date", part="id", maxResults=50
    ).execute()

    video_ids = [item["id"]["videoId"] for item in search["items"]]

    stats = youtube.videos().list(
    id=",".join(video_ids),
    part="snippet,statistics,contentDetails"
    ).execute()

    print()


    for v in stats["items"]:
        if isodate.parse_duration(v['contentDetails']['duration']).total_seconds() < 180 and int(v["statistics"].get("viewCount")) > 10000:
            video_data = (
                {
                "title": v["snippet"]["title"],
                "views": int(v["statistics"].get("viewCount")),
                "likes": v["statistics"].get("likeCount"),
                "comments": v["statistics"].get("commentCount"),
                "link": f"https://youtube.com/watch?v={v['id']}",
                "duration_secs": isodate.parse_duration(v['contentDetails']['duration']).total_seconds(),
                "transcript": process_urls([f"https://youtube.com/watch?v={v['id']}"])
                }
            )

            full_data.append(video_data)


    return full_data

In [115]:
get_videos_data("UCYEtmpGBd3jEO1A8sJ4OEMg")

/var/folders/vt/2vrqs8b50f19c5dvl7krlgx80000gp/T/ipykernel_98946/2064951174.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  after = (datetime.utcnow() - timedelta(days=days)).isoformat("T") + "Z"




[1/1] Processing: https://youtube.com/watch?v=pR6DJ1QZDhY


  Transcribing...                                        
Transcribing chunk 1/1...

[1/1] Processing: https://youtube.com/watch?v=9PZRaW8965k


  Transcribing...                                          
Transcribing chunk 1/1...


[{'title': 'Humare culture me...',
  'views': '9264',
  'likes': '242',
  'comments': '2',
  'link': 'https://youtube.com/watch?v=pR6DJ1QZDhY',
  'duration_secs': 17.0,
  'transcript': {'https://youtube.com/watch?v=pR6DJ1QZDhY': 'हमारे कल्चर में, हमें ये सिखाया नहीं गया है कि इफ आई कैन डिसअग्री, कैन आई डिसअग्री विथ रेस्पेक्ट? एंड इफ आई डिसअग्री टू यू डजंट मीन आई रेस्पेक्ट यू लेस। ये कम्युनिकेट करना इज सच अ पिवेटल पॉइंट इन अ फैमिली बिजनेस।'}},
 {'title': 'Boring makes money.. Entertaining keeps you in struggle..',
  'views': '14622',
  'likes': '404',
  'comments': '3',
  'link': 'https://youtube.com/watch?v=9PZRaW8965k',
  'duration_secs': 20.0,
  'transcript': {'https://youtube.com/watch?v=9PZRaW8965k': 'बिजनेस का रूल क्या है? बोरिंग मेक्स मनी। एंटरटेनिंग कीप्स यू इन स्ट्रगल। मोस्ट पीपल आर एडिक्टेड टू द एंटरटेनमेंट ऑफ स्ट्रगल। उनको बैठ के एसओपी बनाना, ट्रैकर्स बनाना जहां पर एफर्ट डेटा और रिजल्ट डेटा कैप्चर हो, ये उनको करना अच्छा नहीं लगता।'}}]

In [155]:
def full_video_data(list_creators):
    youtube = build("youtube", "v3", developerKey=API_KEY)
    all_data = []
    
    for creator in list_creators:
        channel_id = find_channel_id(creator)
        creator_data = get_videos_data(channel_id)

        all_data.append({
            'channel_id': creator,
            'creator_data': creator_data
        })

    return all_data

In [157]:
creators = ["@daminitripathi"]

full_video_data(creators)

/var/folders/vt/2vrqs8b50f19c5dvl7krlgx80000gp/T/ipykernel_98946/978370315.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  after = (datetime.utcnow() - timedelta(days=days)).isoformat("T") + "Z"




[1/1] Processing: https://youtube.com/watch?v=IVaplXYbJXQ


  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...

[1/1] Processing: https://youtube.com/watch?v=_iQ6S8anTo0


  Transcribing...                                        
Transcribing chunk 1/2...
Transcribing chunk 2/2...


[{'channel_id': '@daminitripathi',
  'creator_data': [{'title': 'Starting Ecommerce? Watch This First ⚡',
    'views': 29569,
    'likes': '2331',
    'comments': '6',
    'link': 'https://youtube.com/watch?v=IVaplXYbJXQ',
    'duration_secs': 56.0,
    'transcript': {'https://youtube.com/watch?v=IVaplXYbJXQ': 'हे हे दामिनी, यार मैं ई-कॉमर्स बिजनेस शुरू करने का सोच रहा हूं और चाइनीस वेबसाइट से मुझे ना भरोसा नहीं होता है तो मैं सामान कहां से सोर्स करूं? चार ऐसे Instagram अकाउंट्स बताती हूं जहां पर चाइनीस क्रिएटर्स खुद बता रहे हैं कि सामान कहां से और कैसे सोर्स करना है। टीना चाइना सोर्सिंग अकाउंट को फॉलो कर लो बिकॉज़ ये लिटरली हर एक वेयरहाउस में जाकर बताती है कि आपको प्रोडक्ट का प्राइस कितना पड़ेगा और अगर आप इसे अपनी कंट्री में मंगवाते हो तो आपको एक्चुअल कॉस्ट कितना पड़ने वाला है। ये विंचू वाले अकाउंट को फॉलो कर लेना बिकॉज़ अगर आपको ₹18 के अंदर है ना तो उसकी पूरी वेयरहाउसिंग इनके पास मिल जाएगी एंड ये इवन आपको इंडिया में स्टोर भी सेटअप करके दे देंगे। थर्ड है कारा चाइना सोर्सिंग आप इनके अक

In [167]:
with open("/Users/Rakshit.Lodha/Downloads/youtube_channels_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [209]:
data

[[{'channel_id': '@Rajiv.Talreja',
   'creator_data': [{'title': 'Humare culture me...',
     'views': '9264',
     'likes': '242',
     'comments': '2',
     'link': 'https://youtube.com/watch?v=pR6DJ1QZDhY',
     'duration_secs': 17.0,
     'transcript': {'https://youtube.com/watch?v=pR6DJ1QZDhY': 'हमारे कल्चर में, हमें ये सिखाया नहीं गया है कि इफ आई कैन डिसअग्री, कैन आई डिसअग्री विथ रेस्पेक्ट? एंड इफ आई डिसअग्री टू यू डजंट मीन आई रेस्पेक्ट यू लेस। ये कम्युनिकेट करना इज सच अ पिवेटल पॉइंट इन अ फैमिली बिजनेस।'}},
    {'title': 'Boring makes money.. Entertaining keeps you in struggle..',
     'views': '14622',
     'likes': '404',
     'comments': '3',
     'link': 'https://youtube.com/watch?v=9PZRaW8965k',
     'duration_secs': 20.0,
     'transcript': {'https://youtube.com/watch?v=9PZRaW8965k': 'बिजनेस का रूल क्या है? बोरिंग मेक्स मनी। एंटरटेनिंग कीप्स यू इन स्ट्रगल। मोस्ट पीपल आर एडिक्टेड टू द एंटरटेनमेंट ऑफ स्ट्रगल। उनको बैठ के एसओपी बनाना, ट्रैकर्स बनाना जहां पर एफर्ट डेटा और रिजल्

In [210]:
ANALYSIS_TOOL = {
    "name": "save_video_analysis",
    "description": "Save the topic analysis of a YouTube short video",
    "input_schema": {
        "type": "object",
        "properties": {
            "primary_topic": {
                "type": "string",
                "description": "Main topic in 2-3 words"
            },
            "sub_topics": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of subtopics covered, 2-3 words each"
            },
            "content_format": {
                "type": "string",
                "description": "Format/style of the video in 2-3 words"
            },
            "language": {
                "type": "string",
                "enum": ["hindi", "hinglish", "english"]
            }
        },
        "required": ["primary_topic", "sub_topics", "content_format", "language"]
    }
}

In [222]:
def analyze_video(video):
    transcript_text = video['transcript']
    if isinstance(transcript_text, dict):
        transcript_text = next(iter(transcript_text.values()))

    resp = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        tools=[ANALYSIS_TOOL_1],
        tool_choice={"type": "tool", "name": "save_video_analysis"},
        messages=[{"role": "user", "content": f"""Analyze this YouTube short:

Title: {video['title']}
Views: {video['views']}
Likes: {video['likes']}
Duration: {video['duration_secs']}s
Transcript: {transcript_text}"""}]
    )

    for block in resp.content:
        if block.type == "tool_use":
            return block.input
    return None

In [217]:
enriched_data = []

for creator_block in data:
    for creator in creator_block:
        print(f"\n📺 {creator['channel_id']}")
        for video in creator['creator_data']:
            tags = analyze_video(video)
            enriched_data.append({
                "creator": creator['channel_id'],
                "title": video['title'],
                "views": int(video['views']),
                "likes": int(video['likes']),
                "comments": int(video['comments']),
                "duration_secs": video['duration_secs'],
                "link": video['link'],
                "tags": tags
            })
            print(f"  ✓ {video['title'][:50]}...")


📺 @Rajiv.Talreja
  ✓ Humare culture me......
  ✓ Boring makes money.. Entertaining keeps you in str...

📺 @ankitravindrajain
  ✓ Stop Doing THIS in Sales Meetings 🤯(97% of Salespe...
  ✓ Master the Power of Body Language #shorts #bodylan...
  ✓ How to ANSWER It’s Too Expensive Sales Objection 🤯...
  ✓ How to Grow your Offline Business SALES 🚀#salestip...
  ✓ How to Handle Price Objections in high-ticket sale...
  ✓ How to Sell a Table Tennis for ₹35,000🔥 #shorts #s...
  ✓ How to Handle "I'll Get Back to You" Sales Objecti...
  ✓ How to Overcome Sales Objections 🔥#shorts #salesti...
  ✓ How to Sell a ₹200 Bottle for ₹5,000 🚀 #shorts #sa...
  ✓ How to Answer ANY Sales Objection in 10 Seconds 🔥#...
  ✓ Sales Meeting Body Language: Why 90% of Salespeopl...

📺 @SanjayNuthraYT
  ✓ Alex Hormozi didn’t start content to make money he...

📺 @MrVivekBindra
  ✓ Can small investments really create big profits?...
  ✓ हनुमान जी ने कैसे उठाया पूरा पर्वत? #shorts #drviv...
  ✓ जब आरती खेतरपाल जी ने ग

In [218]:
enriched_data

[{'creator': '@Rajiv.Talreja',
  'title': 'Humare culture me...',
  'views': 9264,
  'likes': 242,
  'comments': 2,
  'duration_secs': 17.0,
  'link': 'https://youtube.com/watch?v=pR6DJ1QZDhY',
  'tags': {'primary_topic': 'Family Business',
   'sub_topics': ['Communication Skills',
    'Respectful Disagreement',
    'Cultural Values',
    'Business Relationships'],
   'content_format': 'Advice/Commentary',
   'language': 'hinglish'}},
 {'creator': '@Rajiv.Talreja',
  'title': 'Boring makes money.. Entertaining keeps you in struggle..',
  'views': 14622,
  'likes': 404,
  'comments': 3,
  'duration_secs': 20.0,
  'link': 'https://youtube.com/watch?v=9PZRaW8965k',
  'tags': {'primary_topic': 'Business Strategy',
   'sub_topics': ['Boring vs Entertaining',
    'SOPs and Systems',
    'Data Tracking',
    'Success Mindset'],
   'content_format': 'Motivational advice',
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'Stop Doing THIS in Sales Meetings 🤯(97% of Sale

In [219]:
import pandas as pd

In [220]:
df = pd.json_normalize(enriched_data)
df.to_csv("enriched_data.csv", index=False)

In [221]:
ANALYSIS_TOOL_1 = {
    "name": "save_video_analysis",
    "description": "Save the topic analysis of a YouTube short video",
    "input_schema": {
        "type": "object",
        "properties": {
            "primary_topic": {
                "type": "string",
                "description": "Main topic in 2-3 words"
            },
            "topic_category": {
                "type": "string",
                "enum": [
                    "sales & negotiation",
                    "business strategy",
                    "marketing",
                    "investing & finance",
                    "ecommerce",
                    "mythology & spirituality",
                    "motivational story",
                    "celebrity & pop culture",
                    "history & biography",
                    "tech & science",
                    "personal development",
                    "content creation"
                ]
            },
            "content_format": {
                "type": "string",
                "enum": [
                    "roleplay/skit",
                    "talking head",
                    "storytelling",
                    "interview/podcast clip",
                    "tutorial/listicle",
                    "devotional/chant",
                    "motivational monologue"
                ]
            },
            "sub_topics": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of subtopics covered, 2-3 words each"
            },
            "language": {
                "type": "string",
                "enum": ["hindi", "hinglish", "english"]
            }
        },
        "required": ["primary_topic", "topic_category", "content_format", "sub_topics", "language"]
    }
}

In [223]:
enriched_data_1 = []

for creator_block in data:
    for creator in creator_block:
        print(f"\n📺 {creator['channel_id']}")
        for video in creator['creator_data']:
            tags = analyze_video(video)
            enriched_data.append({
                "creator": creator['channel_id'],
                "title": video['title'],
                "views": int(video['views']),
                "likes": int(video['likes']),
                "comments": int(video['comments']),
                "duration_secs": video['duration_secs'],
                "link": video['link'],
                "tags": tags
            })
            print(f"  ✓ {video['title'][:50]}...")


📺 @Rajiv.Talreja
  ✓ Humare culture me......
  ✓ Boring makes money.. Entertaining keeps you in str...

📺 @ankitravindrajain
  ✓ Stop Doing THIS in Sales Meetings 🤯(97% of Salespe...
  ✓ Master the Power of Body Language #shorts #bodylan...
  ✓ How to ANSWER It’s Too Expensive Sales Objection 🤯...
  ✓ How to Grow your Offline Business SALES 🚀#salestip...
  ✓ How to Handle Price Objections in high-ticket sale...
  ✓ How to Sell a Table Tennis for ₹35,000🔥 #shorts #s...
  ✓ How to Handle "I'll Get Back to You" Sales Objecti...
  ✓ How to Overcome Sales Objections 🔥#shorts #salesti...
  ✓ How to Sell a ₹200 Bottle for ₹5,000 🚀 #shorts #sa...
  ✓ How to Answer ANY Sales Objection in 10 Seconds 🔥#...
  ✓ Sales Meeting Body Language: Why 90% of Salespeopl...

📺 @SanjayNuthraYT
  ✓ Alex Hormozi didn’t start content to make money he...

📺 @MrVivekBindra
  ✓ Can small investments really create big profits?...
  ✓ हनुमान जी ने कैसे उठाया पूरा पर्वत? #shorts #drviv...
  ✓ जब आरती खेतरपाल जी ने ग

In [225]:
enriched_data

[{'creator': '@Rajiv.Talreja',
  'title': 'Humare culture me...',
  'views': 9264,
  'likes': 242,
  'comments': 2,
  'duration_secs': 17.0,
  'link': 'https://youtube.com/watch?v=pR6DJ1QZDhY',
  'tags': {'primary_topic': 'Family Business',
   'sub_topics': ['Communication Skills',
    'Respectful Disagreement',
    'Cultural Values',
    'Business Relationships'],
   'content_format': 'Advice/Commentary',
   'language': 'hinglish'}},
 {'creator': '@Rajiv.Talreja',
  'title': 'Boring makes money.. Entertaining keeps you in struggle..',
  'views': 14622,
  'likes': 404,
  'comments': 3,
  'duration_secs': 20.0,
  'link': 'https://youtube.com/watch?v=9PZRaW8965k',
  'tags': {'primary_topic': 'Business Strategy',
   'sub_topics': ['Boring vs Entertaining',
    'SOPs and Systems',
    'Data Tracking',
    'Success Mindset'],
   'content_format': 'Motivational advice',
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'Stop Doing THIS in Sales Meetings 🤯(97% of Sale

In [226]:
a = [{'creator': '@Rajiv.Talreja',
  'title': 'Humare culture me...',
  'views': 9264,
  'likes': 242,
  'comments': 2,
  'duration_secs': 17.0,
  'link': 'https://youtube.com/watch?v=pR6DJ1QZDhY',
  'tags': {'primary_topic': 'respectful disagreement',
   'topic_category': 'business strategy',
   'content_format': 'talking head',
   'sub_topics': ['family business',
    'communication skills',
    'cultural values',
    'respect in business'],
   'language': 'hinglish'}},
 {'creator': '@Rajiv.Talreja',
  'title': 'Boring makes money.. Entertaining keeps you in struggle..',
  'views': 14622,
  'likes': 404,
  'comments': 3,
  'duration_secs': 20.0,
  'link': 'https://youtube.com/watch?v=9PZRaW8965k',
  'tags': {'primary_topic': 'business discipline',
   'topic_category': 'business strategy',
   'content_format': 'talking head',
   'sub_topics': ['SOPs and systems',
    'data tracking',
    'avoiding entertainment',
    'struggle mindset'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'Stop Doing THIS in Sales Meetings 🤯(97% of Salespeople Doing It) #shorts #salestips #salestraining',
  'views': 14983,
  'likes': 1197,
  'comments': 58,
  'duration_secs': 158.0,
  'link': 'https://youtube.com/watch?v=UNnWRDCdw70',
  'tags': {'primary_topic': 'Sales meetings',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['preparation mistakes',
    'pricing strategy',
    'closing techniques',
    'professionalism',
    'sales presentation'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'Master the Power of Body Language #shorts #bodylanguage #ytshorts',
  'views': 10615,
  'likes': 106,
  'comments': 0,
  'duration_secs': 8.0,
  'link': 'https://youtube.com/watch?v=uuyn8vh8pvs',
  'tags': {'primary_topic': 'Body Language',
   'topic_category': 'personal development',
   'content_format': 'motivational monologue',
   'sub_topics': ['communication skills',
    'non-verbal cues',
    'self-improvement'],
   'language': 'english'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to ANSWER It’s Too Expensive Sales Objection 🤯 #shorts #salestips #salestraining #ytshorts',
  'views': 10050,
  'likes': 594,
  'comments': 8,
  'duration_secs': 81.0,
  'link': 'https://youtube.com/watch?v=gytJUs8AR2g',
  'tags': {'primary_topic': 'price objection handling',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['compare to what',
    'avoid discounting',
    'calculate opportunity cost',
    'value comparison',
    'closing technique'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Grow your Offline Business SALES 🚀#salestips #sales #salestraining',
  'views': 13361,
  'likes': 1044,
  'comments': 80,
  'duration_secs': 133.0,
  'link': 'https://youtube.com/watch?v=REbRC2A7lPA',
  'tags': {'primary_topic': 'offline business growth',
   'topic_category': 'sales & negotiation',
   'content_format': 'talking head',
   'sub_topics': ['decommoditization strategy',
    'no-brainer offer',
    'competitive pricing',
    'value bundling',
    'customer service'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Handle Price Objections in high-ticket sales 🚀 #salestips #shorts #salestraining',
  'views': 40076,
  'likes': 1738,
  'comments': 28,
  'duration_secs': 100.0,
  'link': 'https://youtube.com/watch?v=Zq3X6ihMahU',
  'tags': {'primary_topic': 'Price objection handling',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['value-based selling',
    'no-brainer offer',
    'free delivery',
    'free installation',
    'extended support',
    'high-ticket sales'],
   'language': 'hindi'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Sell a Table Tennis for ₹35,000🔥 #shorts #salestips #salestraining',
  'views': 2886678,
  'likes': 76904,
  'comments': 619,
  'duration_secs': 97.0,
  'link': 'https://youtube.com/watch?v=zzt0-7CdtCQ',
  'tags': {'primary_topic': 'Table tennis sales',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['qualifying customer',
    'product positioning',
    'tiered pricing strategy',
    'understanding needs',
    'closing sales'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Handle "I\'ll Get Back to You" Sales Objection 🔥#salestips #salestraining #salesobjections',
  'views': 25167,
  'likes': 2434,
  'comments': 199,
  'duration_secs': 101.0,
  'link': 'https://youtube.com/watch?v=nlNv3iG-m20',
  'tags': {'primary_topic': 'sales objection handling',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ["I'll get back objection",
    'uncovering real objections',
    'price objections',
    'product objections',
    'sales conversion techniques'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Overcome Sales Objections 🔥#shorts #salestips #salestraining',
  'views': 17825,
  'likes': 1193,
  'comments': 72,
  'duration_secs': 125.0,
  'link': 'https://youtube.com/watch?v=uRKM95BUIjI',
  'tags': {'primary_topic': 'Sales objections',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['price negotiation',
    'closing techniques',
    'objection handling',
    'decision-making concerns',
    'sales masterclass promotion'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Sell a ₹200 Bottle for ₹5,000 🚀 #shorts #salestips #salestraining',
  'views': 196543,
  'likes': 4133,
  'comments': 61,
  'duration_secs': 112.0,
  'link': 'https://youtube.com/watch?v=z19NMfmm2dk',
  'tags': {'primary_topic': 'Selling technique',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['value-based selling',
    'brand image positioning',
    'sales demonstration',
    'price justification'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'How to Answer ANY Sales Objection in 10 Seconds 🔥#salestips #salestraining',
  'views': 119864,
  'likes': 3862,
  'comments': 21,
  'duration_secs': 89.0,
  'link': 'https://youtube.com/watch?v=71unyUdIlGA',
  'tags': {'primary_topic': 'sales objections',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['objection handling',
    'closing techniques',
    'sales responses',
    'customer concerns'],
   'language': 'hinglish'}},
 {'creator': '@ankitravindrajain',
  'title': 'Sales Meeting Body Language: Why 90% of Salespeople Fail in the First 30 Seconds #salestips #sales',
  'views': 11022,
  'likes': 572,
  'comments': 97,
  'duration_secs': 79.0,
  'link': 'https://youtube.com/watch?v=0BXfQaLw_D0',
  'tags': {'primary_topic': 'Body Language Sales',
   'topic_category': 'sales & negotiation',
   'content_format': 'roleplay/skit',
   'sub_topics': ['first impressions',
    'sales meetings',
    'sample presentation',
    'price negotiation',
    'masterclass promotion'],
   'language': 'hinglish'}},
 {'creator': '@SanjayNuthraYT',
  'title': 'Alex Hormozi didn’t start content to make money he started it for long-term leverage. #shorts',
  'views': 16555,
  'likes': 385,
  'comments': 1,
  'duration_secs': 31.0,
  'link': 'https://youtube.com/watch?v=aSCVwZ3mnds',
  'tags': {'primary_topic': 'Alex Hormozi strategy',
   'topic_category': 'business strategy',
   'content_format': 'talking head',
   'sub_topics': ['content creation leverage',
    'long-term vision',
    'billionaire mindset',
    'coaching business'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Can small investments really create big profits?',
  'views': 11149,
  'likes': 314,
  'comments': 3,
  'duration_secs': 60.0,
  'link': 'https://youtube.com/watch?v=uKKT454gAMk',
  'tags': {'primary_topic': 'SIP investments',
   'topic_category': 'investing & finance',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['systematic investment plan',
    'wealth building strategies',
    'compound growth',
    'SWP withdrawals',
    'EMI planning'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'हनुमान जी ने कैसे उठाया पूरा पर्वत? #shorts #drvivekbindra',
  'views': 23160,
  'likes': 669,
  'comments': 14,
  'duration_secs': 35.0,
  'link': 'https://youtube.com/watch?v=8H_l_-ErWN0',
  'tags': {'primary_topic': 'Hanuman mythology',
   'topic_category': 'mythology & spirituality',
   'content_format': 'roleplay/skit',
   'sub_topics': ['Ravana character',
    'Hanuman strength',
    'Sanjeevani mountain',
    'mythological story'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'जब आरती खेतरपाल जी ने गाया कृष्ण भजन… पूरा माहौल हो गया दिव्य #shorts #drvivekbindra',
  'views': 21334,
  'likes': 351,
  'comments': 17,
  'duration_secs': 52.0,
  'link': 'https://youtube.com/watch?v=K7mI1fDA5YE',
  'tags': {'primary_topic': 'Krishna Bhajan',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['Aarti Khetrapal',
    'Radha Krishna',
    'devotional singing',
    'spiritual atmosphere'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Radhe-Radhe or Radhe-Krishna, which is the right way to chant?',
  'views': 18475,
  'likes': 505,
  'comments': 8,
  'duration_secs': 71.0,
  'link': 'https://youtube.com/watch?v=6qOBwPf0zBs',
  'tags': {'primary_topic': 'Radha chanting debate',
   'topic_category': 'mythology & spirituality',
   'content_format': 'talking head',
   'sub_topics': ['Radhe-Radhe vs Radhe-Krishna',
    'Hare Krishna mantra',
    'Radha naam significance',
    'spiritual chanting practices',
    'guru mantra guidance'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'When divine energies come together, a celebration becomes truly meaningful. #shorts #drvivekbindra',
  'views': 12567,
  'likes': 438,
  'comments': 16,
  'duration_secs': 53.0,
  'link': 'https://youtube.com/watch?v=xGDnD8SDrzo',
  'tags': {'primary_topic': 'divine celebration',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['divine energies',
    'meaningful celebration',
    'spiritual gathering'],
   'language': 'english'}},
 {'creator': '@MrVivekBindra',
  'title': 'Ye Decision Aapki Zindagi Bacha Sakta Hai',
  'views': 22271,
  'likes': 789,
  'comments': 6,
  'duration_secs': 58.0,
  'link': 'https://youtube.com/watch?v=j1OkjKQ9qPI',
  'tags': {'primary_topic': 'Decision Making',
   'topic_category': 'personal development',
   'content_format': 'storytelling',
   'sub_topics': ['direction vs time',
    'life choices',
    'focus strategy',
    'problem solving'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'When music turns into devotion, magic like this is created... #shorts #drvivekbindra',
  'views': 17714,
  'likes': 450,
  'comments': 5,
  'duration_secs': 61.0,
  'link': 'https://youtube.com/watch?v=9gZkcSKO2Kk',
  'tags': {'primary_topic': 'devotional chanting',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['Hari chant', 'spiritual music', 'devotional performance'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Aaj Samajh Aaya Log Manish Bhai Ko Itna Kyun Chahte Hain! JJ Communications #shorts #drvivekbindra',
  'views': 23115,
  'likes': 350,
  'comments': 8,
  'duration_secs': 35.0,
  'link': 'https://youtube.com/watch?v=cF_1Tvg7PaE',
  'tags': {'primary_topic': 'Manish appreciation',
   'topic_category': 'motivational story',
   'content_format': 'talking head',
   'sub_topics': ['Punjabi poetry', 'personality traits', 'social respect'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Sabse Bada Khan Kaun? Mera Answer Sunke Aap Hairaan Ho Jaoge! #shorts #drvivekbindra #khansir',
  'views': 25644,
  'likes': 576,
  'comments': 13,
  'duration_secs': 36.0,
  'link': 'https://youtube.com/watch?v=AjenzkjIhKA',
  'tags': {'primary_topic': 'Greatest Khan',
   'topic_category': 'celebrity & pop culture',
   'content_format': 'talking head',
   'sub_topics': ['Bollywood comparison', 'Khan actors', 'surprising answer'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'India’s Pride The Great Khali Joins Dr Vivek Bindra’s Grand Celebration #khali #shorts #shortvideo',
  'views': 28282,
  'likes': 542,
  'comments': 7,
  'duration_secs': 39.0,
  'link': 'https://youtube.com/watch?v=PIMoYJTRPl8',
  'tags': {'primary_topic': 'celebrity appearance',
   'topic_category': 'celebrity & pop culture',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['Great Khali',
    'Dr Vivek Bindra',
    'celebration event',
    'cultural song'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Not Just Friendship.. It’s Brotherhood ❤️ | Vivek Oberoi at Dr Vivek Bindra’s Birthday #shorts',
  'views': 26214,
  'likes': 411,
  'comments': 16,
  'duration_secs': 30.0,
  'link': 'https://youtube.com/watch?v=CdyzWCmEI6Y',
  'tags': {'primary_topic': 'friendship and brotherhood',
   'topic_category': 'personal development',
   'content_format': 'motivational monologue',
   'sub_topics': ['celebrity friendship',
    'brotherhood bond',
    'birthday celebration'],
   'language': 'english'}},
 {'creator': '@MrVivekBindra',
  'title': 'हनुमान जी के बचपन की ये लीला हर किसी को चौंका देगी #shorts #shortvideo #hanumanji #drvivekbindra',
  'views': 41845,
  'likes': 1563,
  'comments': 32,
  'duration_secs': 103.0,
  'link': 'https://youtube.com/watch?v=ZAKM1Av0jCE',
  'tags': {'primary_topic': 'Hanuman childhood story',
   'topic_category': 'mythology & spirituality',
   'content_format': 'storytelling',
   'sub_topics': ['Hanuman eating sun',
    'Divine boons',
    "Indra's vajra",
    'Hidden powers',
    'Ramayana reference'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Taj Hotel पर हमला… और Ratan Tata का ऐतिहासिक जवाब #shorts #shortvideo #drvivekbindra',
  'views': 33252,
  'likes': 1490,
  'comments': 21,
  'duration_secs': 85.0,
  'link': 'https://youtube.com/watch?v=73cdhBmVGaY',
  'tags': {'primary_topic': 'Ratan Tata',
   'topic_category': 'history & biography',
   'content_format': 'storytelling',
   'sub_topics': ['Taj Hotel attack',
    '26/11 Mumbai',
    'business ethics',
    'patriotism'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'अहंकार का ऐसा अंत आपने कभी नहीं देखा होगा! #shorts #shortvideo #drvivekbindra #shortstory',
  'views': 38221,
  'likes': 1104,
  'comments': 14,
  'duration_secs': 70.0,
  'link': 'https://youtube.com/watch?v=JkF09Undm8M',
  'tags': {'primary_topic': 'Bhasmasura story',
   'topic_category': 'mythology & spirituality',
   'content_format': 'storytelling',
   'sub_topics': ['ego destruction',
    'Lord Shiva',
    'Lord Vishnu',
    'Mohini avatar',
    'divine lesson'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Is the calendar we follow actually wrong? Kuldeep Singhania | #shorts #shortsvideo #drvivekbindra',
  'views': 28847,
  'likes': 704,
  'comments': 12,
  'duration_secs': 68.0,
  'link': 'https://youtube.com/watch?v=D_W5ugwFObY',
  'tags': {'primary_topic': 'Calendar conspiracy',
   'topic_category': 'history & biography',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['13-month calendar',
    'moon orbit cycle',
    'month naming origins',
    'missing dates history',
    'Julius Caesar',
    'August Caesar'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Are you following the trend… or fighting it? #shorts #shortvideo #deepakwadhwa #drvivekbindra',
  'views': 13056,
  'likes': 369,
  'comments': 7,
  'duration_secs': 45.0,
  'link': 'https://youtube.com/watch?v=q9PpDGUkgEo',
  'tags': {'primary_topic': 'Portfolio rebalancing',
   'topic_category': 'investing & finance',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['trend following',
    'company performance',
    'investment strategy',
    'profit maximization',
    'emotional investing'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'जब तपस्या में लीन थे Gautam Buddha… और 108 घोंघों ने दे दिया अपना जीवन! #shorts #shortvideo',
  'views': 11503,
  'likes': 314,
  'comments': 5,
  'duration_secs': 42.0,
  'link': 'https://youtube.com/watch?v=NSCmNSrQuSQ',
  'tags': {'primary_topic': 'Gautam Buddha',
   'topic_category': 'mythology & spirituality',
   'content_format': 'storytelling',
   'sub_topics': ['snail sacrifice',
    "Buddha's meditation",
    'curly hair symbolism',
    'compassion and sacrifice'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Is Elon Musk really preparing to leave Earth? #shorts #shortvideo #drvivekbindra',
  'views': 45250,
  'likes': 1248,
  'comments': 10,
  'duration_secs': 58.0,
  'link': 'https://youtube.com/watch?v=fHsez_-673g',
  'tags': {'primary_topic': 'Elon Musk conspiracy',
   'topic_category': 'celebrity & pop culture',
   'content_format': 'talking head',
   'sub_topics': ['Mars mission',
    'Tesla innovations',
    'SpaceX purpose',
    'AI and robotics',
    'Earth destruction theory'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'जय श्री राम vs जय सियाराम – फर्क क्या है? 99% लोग नहीं जानते! #rasrajjimaharaj #shorts #shortvideo',
  'views': 31110,
  'likes': 715,
  'comments': 17,
  'duration_secs': 41.0,
  'link': 'https://youtube.com/watch?v=WkjRG7tFSsU',
  'tags': {'primary_topic': 'Jai Shri Ram',
   'topic_category': 'mythology & spirituality',
   'content_format': 'talking head',
   'sub_topics': ['Hanuman chants',
    'Sunderkand significance',
    'religious slogans',
    'Ramayana story'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'भारत का सबसे बेखौफ जनरल - Sam Manekshaw Story Hindi #shorts #shortvideo #drvivekbindra',
  'views': 17481,
  'likes': 639,
  'comments': 2,
  'duration_secs': 56.0,
  'link': 'https://youtube.com/watch?v=vpxqu9_-qG8',
  'tags': {'primary_topic': 'Sam Manekshaw',
   'topic_category': 'history & biography',
   'content_format': 'motivational monologue',
   'sub_topics': ['Indian Army General',
    'military leadership',
    'courage and fearlessness',
    'decorated officer',
    'historical figure'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Honeyy Katiyal | Still guessing in real estate while others are multiplying their wealth? #shorts',
  'views': 11088,
  'likes': 232,
  'comments': 0,
  'duration_secs': 48.0,
  'link': 'https://youtube.com/watch?v=eoz6Ml16jjI',
  'tags': {'primary_topic': 'real estate investing',
   'topic_category': 'investing & finance',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['Dubai attacks',
    'wealth building',
    'broker vs developer',
    'real estate strategy',
    'crorepati formula'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'इतिहास का सबसे खतरनाक हथियार: विषकन्याएं!',
  'views': 26269,
  'likes': 658,
  'comments': 8,
  'duration_secs': 33.0,
  'link': 'https://youtube.com/watch?v=xJOLRZSS5xE',
  'tags': {'primary_topic': 'विषकन्याएं',
   'topic_category': 'history & biography',
   'content_format': 'storytelling',
   'sub_topics': ['चाणक्य', 'प्राचीन हथियार', 'राजाओं की हत्या'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Ujjain Mahakal Darshan : सिर्फ यात्रा नहीं, एक दिव्य अनुभूति #shorts #ujjainmahakal #drvivekbindra',
  'views': 20857,
  'likes': 1418,
  'comments': 24,
  'duration_secs': 47.0,
  'link': 'https://youtube.com/watch?v=uLAFzLK7GS8',
  'tags': {'primary_topic': 'Mahakal Darshan',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['Ujjain temple', 'Shiva worship', 'spiritual experience'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Not Just Acting… Kapil Sharma Lives Comedy in Real Life! #shorts #kapilsharma #drvivekbindra',
  'views': 23927,
  'likes': 537,
  'comments': 3,
  'duration_secs': 60.0,
  'link': 'https://youtube.com/watch?v=fdjpyroFm8U',
  'tags': {'primary_topic': 'Kapil Sharma',
   'topic_category': 'celebrity & pop culture',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['acting challenges',
    'romance scenes',
    'comfort zone',
    'comedy vs romance'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'विश्वास रखो, बजरंगबली सब संभाल लेंगे #shorts #hanumanjayanti #shortvideo',
  'views': 12622,
  'likes': 577,
  'comments': 20,
  'duration_secs': 57.0,
  'link': 'https://youtube.com/watch?v=OxWsckGMIDU',
  'tags': {'primary_topic': 'Hanuman devotion',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ["Lord Rama's blessing",
    "Hanuman's eternal presence",
    'Kali Yuga protection',
    'Hanuman Chalisa chant'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Are you making costly mistakes in the stock market? Deepak Wadhwa #shortvideo #shorts #drvivekbindra',
  'views': 17446,
  'likes': 549,
  'comments': 4,
  'duration_secs': 76.0,
  'link': 'https://youtube.com/watch?v=oJ6AHcwn2OM',
  'tags': {'primary_topic': 'stock market psychology',
   'topic_category': 'investing & finance',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['fear and greed',
    'technical analysis',
    'chart reading',
    'EMA indicators',
    'investor mistakes'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'India’s Real Estate Secrets EXPOSED | Honeyy Katiyal Podcast #shorts #drvivekbindra #shortvideo',
  'views': 15061,
  'likes': 299,
  'comments': 8,
  'duration_secs': 24.0,
  'link': 'https://youtube.com/watch?v=cF9GE8BBrdU',
  'tags': {'primary_topic': 'Real Estate Podcast',
   'topic_category': 'investing & finance',
   'content_format': 'talking head',
   'sub_topics': ['podcast promotion',
    'real estate industry',
    'investment advice'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'ये कहानी आपको अंदर तक हिला देगी Raja Harishchandra #shorts #shortvideo #drvivekbindra',
  'views': 28800,
  'likes': 1178,
  'comments': 14,
  'duration_secs': 178.0,
  'link': 'https://youtube.com/watch?v=GpYQ9nQe7ec',
  'tags': {'primary_topic': 'Raja Harishchandra',
   'topic_category': 'mythology & spirituality',
   'content_format': 'storytelling',
   'sub_topics': ['truth and sacrifice',
    'divine test',
    'devotion and faith',
    'Rishi Vishwamitra',
    'ultimate sacrifice'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'IDEAL Method | Employees Ko Upgrade Karo, Business Ko 3X Grow Karo #shorts #drvivekbindra',
  'views': 12371,
  'likes': 308,
  'comments': 1,
  'duration_secs': 79.0,
  'link': 'https://youtube.com/watch?v=SGas7DZrWsM',
  'tags': {'primary_topic': 'IDEAL Method',
   'topic_category': 'business strategy',
   'content_format': 'talking head',
   'sub_topics': ['manpower development',
    'delegation strategy',
    'task elimination',
    'automation tactics',
    'outsourcing liberation'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'How to Grow FAST Without Ads or Big Budget #shorts #shortvideo #businessstrategy #drvivekbindra',
  'views': 29898,
  'likes': 863,
  'comments': 8,
  'duration_secs': 53.0,
  'link': 'https://youtube.com/watch?v=Y-yjQOEg0m0',
  'tags': {'primary_topic': 'Low-cost marketing',
   'topic_category': 'marketing',
   'content_format': 'talking head',
   'sub_topics': ['viral marketing',
    'Fevicol campaign',
    'unconventional strategy',
    'Bhagavad Gita principle',
    'guerrilla marketing'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Big Marketing Fail? Red Label’s Viral Controversy #shorts #marketingstrategy #drvivekbindra',
  'views': 17389,
  'likes': 809,
  'comments': 4,
  'duration_secs': 111.0,
  'link': 'https://youtube.com/watch?v=sR-NVPxRtSY',
  'tags': {'primary_topic': 'Marketing Controversy',
   'topic_category': 'marketing',
   'content_format': 'talking head',
   'sub_topics': ['Red Label campaign',
    'brand crisis management',
    'cultural sensitivity',
    'quick recovery',
    'business lessons'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Small Village to Global Giant | Zoho Success Story',
  'views': 13932,
  'likes': 432,
  'comments': 2,
  'duration_secs': 44.0,
  'link': 'https://youtube.com/watch?v=dB2tZLlUbzQ',
  'tags': {'primary_topic': 'Zoho Success Story',
   'topic_category': 'business strategy',
   'content_format': 'talking head',
   'sub_topics': ['geo-arbitrage strategy',
    'cost advantage',
    'rural employment model',
    'US market revenue',
    'Fortune 500 clients'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'सब कुछ अपने पास रखना ज़रूरी नहीं होता',
  'views': 40737,
  'likes': 1687,
  'comments': 16,
  'duration_secs': 136.0,
  'link': 'https://youtube.com/watch?v=5w6KuN7qUY8',
  'tags': {'primary_topic': 'letting go',
   'topic_category': 'personal development',
   'content_format': 'storytelling',
   'sub_topics': ['life lessons',
    'moving forward',
    'detachment',
    'motivational metaphor'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Silver vs Gold: The Shocking Truth Investors Must Know!',
  'views': 18630,
  'likes': 721,
  'comments': 6,
  'duration_secs': 84.0,
  'link': 'https://youtube.com/watch?v=ybO6YzBJJgM',
  'tags': {'primary_topic': 'Silver vs Gold',
   'topic_category': 'investing & finance',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['gold-silver ratio',
    'investment opportunity',
    'precious metals',
    'gold prices',
    'silver investment'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Stop Sending Your Kids to School by Bus | Biggest Parenting Mistake?',
  'views': 18364,
  'likes': 626,
  'comments': 7,
  'duration_secs': 50.0,
  'link': 'https://youtube.com/watch?v=_T25mJd2i2U',
  'tags': {'primary_topic': 'School commute advice',
   'topic_category': 'personal development',
   'content_format': 'talking head',
   'sub_topics': ['parenting mistake',
    'time management',
    'school selection',
    "children's wellbeing"],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'खुद की Value नहीं समझी तो ये होगा!',
  'views': 135564,
  'likes': 3857,
  'comments': 14,
  'duration_secs': 137.0,
  'link': 'https://youtube.com/watch?v=wYgW63Qa6wE',
  'tags': {'primary_topic': 'Self Worth',
   'topic_category': 'motivational story',
   'content_format': 'storytelling',
   'sub_topics': ['Value Recognition',
    'Marketing Power',
    'Presentation Matters',
    'Self Assessment'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Biggest Mistakes Every Content Creator Makes..',
  'views': 12962,
  'likes': 394,
  'comments': 6,
  'duration_secs': 47.0,
  'link': 'https://youtube.com/watch?v=drouJarHzPg',
  'tags': {'primary_topic': 'content creator mistakes',
   'topic_category': 'content creation',
   'content_format': 'talking head',
   'sub_topics': ['copying others',
    'unique identity',
    'studying social media',
    'consistent improvement'],
   'language': 'hinglish'}},
 {'creator': '@MrVivekBindra',
  'title': 'Are your private photos really safe… or could they leak anytime?',
  'views': 17482,
  'likes': 766,
  'comments': 2,
  'duration_secs': 82.0,
  'link': 'https://youtube.com/watch?v=JMquzn-MPFw',
  'tags': {'primary_topic': 'Privacy Protection',
   'topic_category': 'tech & science',
   'content_format': 'interview/podcast clip',
   'sub_topics': ['Cybercrime threats',
    'Photo leaks',
    'Legal remedies',
    'Reporting procedures',
    'Online safety'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'चाणक्य की ये कहानी आपको सोचने पर मजबूर कर देगी',
  'views': 20427,
  'likes': 716,
  'comments': 5,
  'duration_secs': 60.0,
  'link': 'https://youtube.com/watch?v=vBqc-gEyLfY',
  'tags': {'primary_topic': 'Chanakya story',
   'topic_category': 'mythology & spirituality',
   'content_format': 'storytelling',
   'sub_topics': ["mother's love",
    'sacrifice',
    'astrology prediction',
    'royal destiny'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Who is greater,  Shri Ram or Shri Krishna?',
  'views': 16868,
  'likes': 704,
  'comments': 17,
  'duration_secs': 66.0,
  'link': 'https://youtube.com/watch?v=T3YG1tbyCFo',
  'tags': {'primary_topic': 'Ram vs Krishna',
   'topic_category': 'mythology & spirituality',
   'content_format': 'talking head',
   'sub_topics': ['avatar differences',
    'Treta vs Dwapara yuga',
    'maryada vs prema',
    'dharma teachings'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Is Barbeque Nation heading towards a downfall?',
  'views': 31393,
  'likes': 1393,
  'comments': 11,
  'duration_secs': 100.0,
  'link': 'https://youtube.com/watch?v=kOhhRz3H384',
  'tags': {'primary_topic': 'Barbeque Nation decline',
   'topic_category': 'business strategy',
   'content_format': 'talking head',
   'sub_topics': ['capital intensive model',
    'debt burden',
    'operational costs',
    'table turnover rate',
    'shrinking margins',
    'pricing strategy',
    'restaurant business'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Navratri 9th Day: ये 1 बात बदल सकती है आपकी किस्मत!',
  'views': 10760,
  'likes': 1099,
  'comments': 18,
  'duration_secs': 39.0,
  'link': 'https://youtube.com/watch?v=OekBs8oKuq4',
  'tags': {'primary_topic': 'Navratri 9th Day',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['Maa Siddhidatri',
    'spiritual powers',
    'Lord Shiva',
    'success mantra'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'राम नवमी 2026: श्रीराम के जीवन से सीखें सफलता और मर्यादा का मंत्र',
  'views': 14299,
  'likes': 386,
  'comments': 12,
  'duration_secs': 27.0,
  'link': 'https://youtube.com/watch?v=6RbQdO6k7RI',
  'tags': {'primary_topic': 'Ram Navami',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['Shri Ram',
    'dharma values',
    'success lessons',
    'moral conduct'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'How did Rajiv Dixit bring Ayurveda back to life?',
  'views': 19373,
  'likes': 996,
  'comments': 7,
  'duration_secs': 69.0,
  'link': 'https://youtube.com/watch?v=s4V1Y3r2A8E',
  'tags': {'primary_topic': 'Rajiv Dixit Ayurveda',
   'topic_category': 'history & biography',
   'content_format': 'talking head',
   'sub_topics': ['Ayurveda revival',
    'Education awareness',
    'Doctor demand increase',
    'Salary growth comparison',
    'Global wellness centers'],
   'language': 'hindi'}},
 {'creator': '@MrVivekBindra',
  'title': 'Navratri Day 8: माँ महागौरी का रहस्य जो आपकी ज़िंदगी बदल सकता है',
  'views': 11763,
  'likes': 1412,
  'comments': 10,
  'duration_secs': 42.0,
  'link': 'https://youtube.com/watch?v=C9B3wbx3txM',
  'tags': {'primary_topic': 'Maa Mahagauri',
   'topic_category': 'mythology & spirituality',
   'content_format': 'devotional/chant',
   'sub_topics': ['Navratri Day 8',
    'Goddess Parvati',
    'worship rituals',
    'divine transformation',
    'spiritual powers'],
   'language': 'hindi'}},
 {'creator': '@daminitripathi',
  'title': 'Starting Ecommerce? Watch This First ⚡',
  'views': 29569,
  'likes': 2331,
  'comments': 6,
  'duration_secs': 56.0,
  'link': 'https://youtube.com/watch?v=IVaplXYbJXQ',
  'tags': {'primary_topic': 'China sourcing',
   'topic_category': 'ecommerce',
   'content_format': 'roleplay/skit',
   'sub_topics': ['Instagram accounts',
    'product sourcing',
    'warehouse tours',
    'import export',
    'manufacturing tips'],
   'language': 'hinglish'}},
 {'creator': '@daminitripathi',
  'title': 'Ecommerce Apps Can Double Your Revenue 📈',
  'views': 13490,
  'likes': 721,
  'comments': 10,
  'duration_secs': 47.0,
  'link': 'https://youtube.com/watch?v=_iQ6S8anTo0',
  'tags': {'primary_topic': 'ecommerce apps',
   'topic_category': 'ecommerce',
   'content_format': 'tutorial/listicle',
   'sub_topics': ['Shopify tools',
    'COD payments',
    'WhatsApp automation',
    'cart abandonment',
    'UGC videos',
    'conversion rate'],
   'language': 'hinglish'}}]

In [227]:
df = pd.json_normalize(a)
df.to_csv("enriched_data_1.csv", index=False)